In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT poppler -> Test on poppler
DANN Domain Adaptation + GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "poppler"
COMBINED_FILE = "../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/dann_gan/poppler"

DANN_EPOCHS      = 50
DANN_BATCH_SIZE   = 64
DOMAIN_LOSS_WEIGHT_MAX = 1.0   # max alpha for domain loss ramp

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator (for minority oversampling)
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Gradient Reversal Layer
# =========================================================
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self):
        super(GradientReversalLayer, self).__init__()

    def forward(self, x, alpha):
        return GradientReversalFunction.apply(x, alpha)


# =========================================================
# Feature Extractor (shared trunk)
# =========================================================
class FeatureExtractor(nn.Module):
    def __init__(self, input_dim):
        super(FeatureExtractor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Classifier Head (vulnerability label predictor)
# =========================================================
class ClassifierHead(nn.Module):
    def __init__(self, feature_dim=16):
        super(ClassifierHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# Domain Discriminator
# =========================================================
class DomainDiscriminator(nn.Module):
    def __init__(self, feature_dim=16):
        super(DomainDiscriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# DANN Training (Phase 1: joint classification + domain adaptation)
# =========================================================
def train_dann(feature_extractor, classifier_head, domain_discriminator,
                x_source, y_source, x_target,
                x_val, y_val, class_weights,
                epochs, batch_size, desc):

    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    grl = GradientReversalLayer()

    params = (list(feature_extractor.parameters()) +
              list(classifier_head.parameters()) +
              list(domain_discriminator.parameters()))
    optimizer = torch.optim.Adam(params, lr=1e-5)

    cls_criterion    = nn.BCELoss(reduction='none')
    domain_criterion = nn.BCELoss()

    x_source_t = torch.tensor(x_source, dtype=torch.float32)
    y_source_t = torch.tensor(y_source, dtype=torch.float32)
    x_target_t = torch.tensor(x_target, dtype=torch.float32)

    source_dataset = TensorDataset(x_source_t, y_source_t)
    source_loader  = DataLoader(source_dataset, batch_size=batch_size, shuffle=True)

    n_batches = len(source_loader)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        domain_discriminator.train()

        target_loader = DataLoader(
            TensorDataset(x_target_t), batch_size=batch_size, shuffle=True
        )
        target_iter = iter(target_loader)

        for batch_idx, (xs_batch, ys_batch) in enumerate(source_loader):
            p = (epoch * n_batches + batch_idx) / (epochs * n_batches)
            alpha = DOMAIN_LOSS_WEIGHT_MAX * (2.0 / (1.0 + np.exp(-10 * p)) - 1.0)

            try:
                (xt_batch,) = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                (xt_batch,) = next(target_iter)

            xs_batch = xs_batch.to(DEVICE)
            ys_batch = ys_batch.to(DEVICE)
            xt_batch = xt_batch.to(DEVICE)

            optimizer.zero_grad()

            # --- Classification loss on source ---
            source_features = feature_extractor(xs_batch)
            y_pred = classifier_head(source_features).squeeze()
            weights = torch.where(ys_batch == 1, pos_weight, neg_weight)
            cls_loss = (cls_criterion(y_pred, ys_batch) * weights).mean()

            # --- Domain loss on source + target ---
            target_features = feature_extractor(xt_batch)

            bsz_s = source_features.size(0)
            bsz_t = target_features.size(0)

            source_reversed = grl(source_features, alpha)
            target_reversed = grl(target_features, alpha)

            domain_pred_source = domain_discriminator(source_reversed)
            domain_pred_target = domain_discriminator(target_reversed)

            domain_labels_source = torch.zeros(bsz_s, 1).to(DEVICE)
            domain_labels_target = torch.ones(bsz_t, 1).to(DEVICE)

            domain_loss = (
                domain_criterion(domain_pred_source, domain_labels_source) +
                domain_criterion(domain_pred_target, domain_labels_target)
            )

            total_loss = cls_loss + domain_loss
            total_loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            val_weights  = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (cls_criterion(val_preds, y_val) * val_weights).mean().item()

        print(f"      {desc} | Epoch {epoch+1}/{epochs} - "
              f"cls_loss: {cls_loss.item():.4f}  domain_loss: {domain_loss.item():.4f}  "
              f"alpha: {alpha:.3f}  val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Fine-tuning Loop (Phase 2: pseudo-labeled, classification only)
# =========================================================
def fine_tune_on_pseudo_labels(feature_extractor, classifier_head,
                                dataloader, x_val, y_val,
                                optimizer, criterion, epochs, desc, class_weights):
    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            features    = feature_extractor(x_batch)
            predictions = classifier_head(features).squeeze()
            weights     = torch.where(y_batch == 1, pos_weight, neg_weight)
            loss        = (criterion(predictions, y_batch) * weights).mean()
            loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            weights      = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (criterion(val_preds, y_val) * weights).mean().item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Semi-supervised Transfer Learning with DANN
# =========================================================
def semi_supervised_transfer_learning_dann(x_train, y_train, x_test,
                                            class_weights, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    feature_extractor     = FeatureExtractor(x_train.shape[1]).to(DEVICE)
    classifier_head       = ClassifierHead(feature_dim=16).to(DEVICE)
    domain_discriminator  = DomainDiscriminator(feature_dim=16).to(DEVICE)

    # Phase 1: DANN training on GAN-oversampled source + target domain alignment
    feature_extractor, classifier_head = train_dann(
        feature_extractor, classifier_head, domain_discriminator,
        x_source=x_tr, y_source=y_tr, x_target=x_test,
        x_val=x_val_t, y_val=y_val_t,
        class_weights=class_weights,
        epochs=DANN_EPOCHS,
        batch_size=DANN_BATCH_SIZE,
        desc=f"Iter {iteration} - Phase 1 (DANN)"
    )

    # Pseudo-label the test set (poppler)
    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features = feature_extractor(x_test_t.to(DEVICE))
        y_pred        = classifier_head(test_features).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source (GAN) + pseudo-labelled poppler
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(
        list(feature_extractor.parameters()) + list(classifier_head.parameters()),
        lr=1e-5
    )
    criterion = nn.BCELoss(reduction='none')

    feature_extractor, classifier_head = fine_tune_on_pseudo_labels(
        feature_extractor, classifier_head,
        dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2",
        class_weights=class_weights
    )

    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features_final = feature_extractor(x_test_t.to(DEVICE))
        y_pred_final = classifier_head(test_features_final).squeeze().cpu().numpy()

    return (feature_extractor, classifier_head), y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, class_weights, sample_weights=None):
    scale_pos_weight = class_weights[1] / class_weights[0]
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (poppler Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except poppler -> Test on poppler (DANN + GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/7] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/7] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (poppler)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/7] Normalize
    # ----------------------------------------------------------
    print("\n[2/7] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/7] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/7] Training GAN on minority class...")
    minority_emb   = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    if n_to_generate <= 0:
        print("      Training pool already balanced or majority-minority; skipping GAN generation.")
        train_emb_gan = train_emb
        train_lbl_gan = train_lbl
    else:
        generator = train_gan(
            minority_emb,
            noise_dim=GAN_NOISE_DIM,
            epochs=GAN_EPOCHS,
            batch_size=GAN_BATCH_SIZE,
            lr=GAN_LR
        )

        synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
        synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

        train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
        train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/7] Class weights (computed on GAN-balanced pool)
    # ----------------------------------------------------------
    print("\n[4/7] Computing class weights...")
    classes           = np.unique(train_lbl_gan)
    class_weights_arr = compute_class_weight(
        class_weight='balanced', classes=classes, y=train_lbl_gan
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
    print(f"      Class weights: {class_weight_dict}")

    # ----------------------------------------------------------
    # [5/7] Ensemble (DANN-NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[5/7] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training DANN neural network...")
        (feature_extractor, classifier_head), _ = semi_supervised_transfer_learning_dann(
            train_emb_gan, train_lbl_gan,
            test_emb,
            class_weight_dict,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            x_tr_t         = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            train_features = feature_extractor(x_tr_t)
            y_train_pred   = classifier_head(train_features).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(
            train_emb_gan, train_lbl_gan, class_weight_dict, sample_weights
        )
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [6/7] Metrics
    # ----------------------------------------------------------
    print("\n[6/7] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (poppler Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [7/7] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"            : "dann_gan",
        "train"                 : "all_except_poppler",
        "test"                  : TEST_PROJ,
        "n_train_projects"      : int(len(set(proj_all[train_mask]))),
        "n_train_samples"       : int(len(train_lbl)),
        "n_train_gan"           : int(len(train_lbl_gan)),
        "n_test_samples"        : int(len(test_lbl)),
        "n_vuln_train"          : int(train_lbl.sum()),
        "n_vuln_train_gan"      : int(train_lbl_gan.sum()),
        "n_vuln_test"           : int(test_lbl.sum()),
        "n_synthetic_generated" : int(n_to_generate),
        "accuracy"              : round(float(accuracy),  4),
        "precision"             : round(float(precision), 4),
        "recall"                : round(float(recall),    4),
        "f1"                    : round(float(f1),        4),
        "auc"                   : round(float(auc),        4),
        "g_mean"                : round(float(g_mean),    4),
        "pf"                    : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "dann_gan_poppler.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_dann_gan_poppler.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT radare2 -> Test on radare2
DANN Domain Adaptation + GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "radare2"
COMBINED_FILE = "../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/dann_gan/radare2"

DANN_EPOCHS      = 50
DANN_BATCH_SIZE   = 64
DOMAIN_LOSS_WEIGHT_MAX = 1.0   # max alpha for domain loss ramp

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator (for minority oversampling)
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Gradient Reversal Layer
# =========================================================
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self):
        super(GradientReversalLayer, self).__init__()

    def forward(self, x, alpha):
        return GradientReversalFunction.apply(x, alpha)


# =========================================================
# Feature Extractor (shared trunk)
# =========================================================
class FeatureExtractor(nn.Module):
    def __init__(self, input_dim):
        super(FeatureExtractor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Classifier Head (vulnerability label predictor)
# =========================================================
class ClassifierHead(nn.Module):
    def __init__(self, feature_dim=16):
        super(ClassifierHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# Domain Discriminator
# =========================================================
class DomainDiscriminator(nn.Module):
    def __init__(self, feature_dim=16):
        super(DomainDiscriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# DANN Training (Phase 1: joint classification + domain adaptation)
# =========================================================
def train_dann(feature_extractor, classifier_head, domain_discriminator,
                x_source, y_source, x_target,
                x_val, y_val, class_weights,
                epochs, batch_size, desc):

    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    grl = GradientReversalLayer()

    params = (list(feature_extractor.parameters()) +
              list(classifier_head.parameters()) +
              list(domain_discriminator.parameters()))
    optimizer = torch.optim.Adam(params, lr=1e-5)

    cls_criterion    = nn.BCELoss(reduction='none')
    domain_criterion = nn.BCELoss()

    x_source_t = torch.tensor(x_source, dtype=torch.float32)
    y_source_t = torch.tensor(y_source, dtype=torch.float32)
    x_target_t = torch.tensor(x_target, dtype=torch.float32)

    source_dataset = TensorDataset(x_source_t, y_source_t)
    source_loader  = DataLoader(source_dataset, batch_size=batch_size, shuffle=True)

    n_batches = len(source_loader)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        domain_discriminator.train()

        target_loader = DataLoader(
            TensorDataset(x_target_t), batch_size=batch_size, shuffle=True
        )
        target_iter = iter(target_loader)

        for batch_idx, (xs_batch, ys_batch) in enumerate(source_loader):
            p = (epoch * n_batches + batch_idx) / (epochs * n_batches)
            alpha = DOMAIN_LOSS_WEIGHT_MAX * (2.0 / (1.0 + np.exp(-10 * p)) - 1.0)

            try:
                (xt_batch,) = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                (xt_batch,) = next(target_iter)

            xs_batch = xs_batch.to(DEVICE)
            ys_batch = ys_batch.to(DEVICE)
            xt_batch = xt_batch.to(DEVICE)

            optimizer.zero_grad()

            # --- Classification loss on source ---
            source_features = feature_extractor(xs_batch)
            y_pred = classifier_head(source_features).squeeze()
            weights = torch.where(ys_batch == 1, pos_weight, neg_weight)
            cls_loss = (cls_criterion(y_pred, ys_batch) * weights).mean()

            # --- Domain loss on source + target ---
            target_features = feature_extractor(xt_batch)

            bsz_s = source_features.size(0)
            bsz_t = target_features.size(0)

            source_reversed = grl(source_features, alpha)
            target_reversed = grl(target_features, alpha)

            domain_pred_source = domain_discriminator(source_reversed)
            domain_pred_target = domain_discriminator(target_reversed)

            domain_labels_source = torch.zeros(bsz_s, 1).to(DEVICE)
            domain_labels_target = torch.ones(bsz_t, 1).to(DEVICE)

            domain_loss = (
                domain_criterion(domain_pred_source, domain_labels_source) +
                domain_criterion(domain_pred_target, domain_labels_target)
            )

            total_loss = cls_loss + domain_loss
            total_loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            val_weights  = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (cls_criterion(val_preds, y_val) * val_weights).mean().item()

        print(f"      {desc} | Epoch {epoch+1}/{epochs} - "
              f"cls_loss: {cls_loss.item():.4f}  domain_loss: {domain_loss.item():.4f}  "
              f"alpha: {alpha:.3f}  val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Fine-tuning Loop (Phase 2: pseudo-labeled, classification only)
# =========================================================
def fine_tune_on_pseudo_labels(feature_extractor, classifier_head,
                                dataloader, x_val, y_val,
                                optimizer, criterion, epochs, desc, class_weights):
    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            features    = feature_extractor(x_batch)
            predictions = classifier_head(features).squeeze()
            weights     = torch.where(y_batch == 1, pos_weight, neg_weight)
            loss        = (criterion(predictions, y_batch) * weights).mean()
            loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            weights      = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (criterion(val_preds, y_val) * weights).mean().item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Semi-supervised Transfer Learning with DANN
# =========================================================
def semi_supervised_transfer_learning_dann(x_train, y_train, x_test,
                                            class_weights, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    feature_extractor     = FeatureExtractor(x_train.shape[1]).to(DEVICE)
    classifier_head       = ClassifierHead(feature_dim=16).to(DEVICE)
    domain_discriminator  = DomainDiscriminator(feature_dim=16).to(DEVICE)

    # Phase 1: DANN training on GAN-oversampled source + target domain alignment
    feature_extractor, classifier_head = train_dann(
        feature_extractor, classifier_head, domain_discriminator,
        x_source=x_tr, y_source=y_tr, x_target=x_test,
        x_val=x_val_t, y_val=y_val_t,
        class_weights=class_weights,
        epochs=DANN_EPOCHS,
        batch_size=DANN_BATCH_SIZE,
        desc=f"Iter {iteration} - Phase 1 (DANN)"
    )

    # Pseudo-label the test set (radare2)
    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features = feature_extractor(x_test_t.to(DEVICE))
        y_pred        = classifier_head(test_features).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source (GAN) + pseudo-labelled radare2
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(
        list(feature_extractor.parameters()) + list(classifier_head.parameters()),
        lr=1e-5
    )
    criterion = nn.BCELoss(reduction='none')

    feature_extractor, classifier_head = fine_tune_on_pseudo_labels(
        feature_extractor, classifier_head,
        dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2",
        class_weights=class_weights
    )

    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features_final = feature_extractor(x_test_t.to(DEVICE))
        y_pred_final = classifier_head(test_features_final).squeeze().cpu().numpy()

    return (feature_extractor, classifier_head), y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, class_weights, sample_weights=None):
    scale_pos_weight = class_weights[1] / class_weights[0]
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (radare2 Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except radare2 -> Test on radare2 (DANN + GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/7] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/7] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (radare2)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/7] Normalize
    # ----------------------------------------------------------
    print("\n[2/7] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/7] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/7] Training GAN on minority class...")
    minority_emb   = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    if n_to_generate <= 0:
        print("      Training pool already balanced or majority-minority; skipping GAN generation.")
        train_emb_gan = train_emb
        train_lbl_gan = train_lbl
    else:
        generator = train_gan(
            minority_emb,
            noise_dim=GAN_NOISE_DIM,
            epochs=GAN_EPOCHS,
            batch_size=GAN_BATCH_SIZE,
            lr=GAN_LR
        )

        synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
        synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

        train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
        train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/7] Class weights (computed on GAN-balanced pool)
    # ----------------------------------------------------------
    print("\n[4/7] Computing class weights...")
    classes           = np.unique(train_lbl_gan)
    class_weights_arr = compute_class_weight(
        class_weight='balanced', classes=classes, y=train_lbl_gan
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
    print(f"      Class weights: {class_weight_dict}")

    # ----------------------------------------------------------
    # [5/7] Ensemble (DANN-NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[5/7] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training DANN neural network...")
        (feature_extractor, classifier_head), _ = semi_supervised_transfer_learning_dann(
            train_emb_gan, train_lbl_gan,
            test_emb,
            class_weight_dict,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            x_tr_t         = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            train_features = feature_extractor(x_tr_t)
            y_train_pred   = classifier_head(train_features).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(
            train_emb_gan, train_lbl_gan, class_weight_dict, sample_weights
        )
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [6/7] Metrics
    # ----------------------------------------------------------
    print("\n[6/7] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (radare2 Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [7/7] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"            : "dann_gan",
        "train"                 : "all_except_radare2",
        "test"                  : TEST_PROJ,
        "n_train_projects"      : int(len(set(proj_all[train_mask]))),
        "n_train_samples"       : int(len(train_lbl)),
        "n_train_gan"           : int(len(train_lbl_gan)),
        "n_test_samples"        : int(len(test_lbl)),
        "n_vuln_train"          : int(train_lbl.sum()),
        "n_vuln_train_gan"      : int(train_lbl_gan.sum()),
        "n_vuln_test"           : int(test_lbl.sum()),
        "n_synthetic_generated" : int(n_to_generate),
        "accuracy"              : round(float(accuracy),  4),
        "precision"             : round(float(precision), 4),
        "recall"                : round(float(recall),    4),
        "f1"                    : round(float(f1),        4),
        "auc"                   : round(float(auc),        4),
        "g_mean"                : round(float(g_mean),    4),
        "pf"                    : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "dann_gan_radare2.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_dann_gan_radare2.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT linux-2.6 -> Test on linux-2.6
DANN Domain Adaptation + GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "linux-2.6"
COMBINED_FILE = "../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/dann_gan/linux-2.6"

DANN_EPOCHS      = 50
DANN_BATCH_SIZE   = 64
DOMAIN_LOSS_WEIGHT_MAX = 1.0   # max alpha for domain loss ramp

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator (for minority oversampling)
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Gradient Reversal Layer
# =========================================================
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self):
        super(GradientReversalLayer, self).__init__()

    def forward(self, x, alpha):
        return GradientReversalFunction.apply(x, alpha)


# =========================================================
# Feature Extractor (shared trunk)
# =========================================================
class FeatureExtractor(nn.Module):
    def __init__(self, input_dim):
        super(FeatureExtractor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Classifier Head (vulnerability label predictor)
# =========================================================
class ClassifierHead(nn.Module):
    def __init__(self, feature_dim=16):
        super(ClassifierHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# Domain Discriminator
# =========================================================
class DomainDiscriminator(nn.Module):
    def __init__(self, feature_dim=16):
        super(DomainDiscriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# DANN Training (Phase 1: joint classification + domain adaptation)
# =========================================================
def train_dann(feature_extractor, classifier_head, domain_discriminator,
                x_source, y_source, x_target,
                x_val, y_val, class_weights,
                epochs, batch_size, desc):

    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    grl = GradientReversalLayer()

    params = (list(feature_extractor.parameters()) +
              list(classifier_head.parameters()) +
              list(domain_discriminator.parameters()))
    optimizer = torch.optim.Adam(params, lr=1e-5)

    cls_criterion    = nn.BCELoss(reduction='none')
    domain_criterion = nn.BCELoss()

    x_source_t = torch.tensor(x_source, dtype=torch.float32)
    y_source_t = torch.tensor(y_source, dtype=torch.float32)
    x_target_t = torch.tensor(x_target, dtype=torch.float32)

    source_dataset = TensorDataset(x_source_t, y_source_t)
    source_loader  = DataLoader(source_dataset, batch_size=batch_size, shuffle=True)

    n_batches = len(source_loader)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        domain_discriminator.train()

        target_loader = DataLoader(
            TensorDataset(x_target_t), batch_size=batch_size, shuffle=True
        )
        target_iter = iter(target_loader)

        for batch_idx, (xs_batch, ys_batch) in enumerate(source_loader):
            p = (epoch * n_batches + batch_idx) / (epochs * n_batches)
            alpha = DOMAIN_LOSS_WEIGHT_MAX * (2.0 / (1.0 + np.exp(-10 * p)) - 1.0)

            try:
                (xt_batch,) = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                (xt_batch,) = next(target_iter)

            xs_batch = xs_batch.to(DEVICE)
            ys_batch = ys_batch.to(DEVICE)
            xt_batch = xt_batch.to(DEVICE)

            optimizer.zero_grad()

            # --- Classification loss on source ---
            source_features = feature_extractor(xs_batch)
            y_pred = classifier_head(source_features).squeeze()
            weights = torch.where(ys_batch == 1, pos_weight, neg_weight)
            cls_loss = (cls_criterion(y_pred, ys_batch) * weights).mean()

            # --- Domain loss on source + target ---
            target_features = feature_extractor(xt_batch)

            bsz_s = source_features.size(0)
            bsz_t = target_features.size(0)

            source_reversed = grl(source_features, alpha)
            target_reversed = grl(target_features, alpha)

            domain_pred_source = domain_discriminator(source_reversed)
            domain_pred_target = domain_discriminator(target_reversed)

            domain_labels_source = torch.zeros(bsz_s, 1).to(DEVICE)
            domain_labels_target = torch.ones(bsz_t, 1).to(DEVICE)

            domain_loss = (
                domain_criterion(domain_pred_source, domain_labels_source) +
                domain_criterion(domain_pred_target, domain_labels_target)
            )

            total_loss = cls_loss + domain_loss
            total_loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            val_weights  = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (cls_criterion(val_preds, y_val) * val_weights).mean().item()

        print(f"      {desc} | Epoch {epoch+1}/{epochs} - "
              f"cls_loss: {cls_loss.item():.4f}  domain_loss: {domain_loss.item():.4f}  "
              f"alpha: {alpha:.3f}  val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Fine-tuning Loop (Phase 2: pseudo-labeled, classification only)
# =========================================================
def fine_tune_on_pseudo_labels(feature_extractor, classifier_head,
                                dataloader, x_val, y_val,
                                optimizer, criterion, epochs, desc, class_weights):
    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            features    = feature_extractor(x_batch)
            predictions = classifier_head(features).squeeze()
            weights     = torch.where(y_batch == 1, pos_weight, neg_weight)
            loss        = (criterion(predictions, y_batch) * weights).mean()
            loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            weights      = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (criterion(val_preds, y_val) * weights).mean().item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Semi-supervised Transfer Learning with DANN
# =========================================================
def semi_supervised_transfer_learning_dann(x_train, y_train, x_test,
                                            class_weights, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    feature_extractor     = FeatureExtractor(x_train.shape[1]).to(DEVICE)
    classifier_head       = ClassifierHead(feature_dim=16).to(DEVICE)
    domain_discriminator  = DomainDiscriminator(feature_dim=16).to(DEVICE)

    # Phase 1: DANN training on GAN-oversampled source + target domain alignment
    feature_extractor, classifier_head = train_dann(
        feature_extractor, classifier_head, domain_discriminator,
        x_source=x_tr, y_source=y_tr, x_target=x_test,
        x_val=x_val_t, y_val=y_val_t,
        class_weights=class_weights,
        epochs=DANN_EPOCHS,
        batch_size=DANN_BATCH_SIZE,
        desc=f"Iter {iteration} - Phase 1 (DANN)"
    )

    # Pseudo-label the test set (linux-2.6)
    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features = feature_extractor(x_test_t.to(DEVICE))
        y_pred        = classifier_head(test_features).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source (GAN) + pseudo-labelled linux-2.6
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(
        list(feature_extractor.parameters()) + list(classifier_head.parameters()),
        lr=1e-5
    )
    criterion = nn.BCELoss(reduction='none')

    feature_extractor, classifier_head = fine_tune_on_pseudo_labels(
        feature_extractor, classifier_head,
        dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2",
        class_weights=class_weights
    )

    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features_final = feature_extractor(x_test_t.to(DEVICE))
        y_pred_final = classifier_head(test_features_final).squeeze().cpu().numpy()

    return (feature_extractor, classifier_head), y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, class_weights, sample_weights=None):
    scale_pos_weight = class_weights[1] / class_weights[0]
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (linux-2.6 Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except linux-2.6 -> Test on linux-2.6 (DANN + GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/7] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/7] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (linux-2.6)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/7] Normalize
    # ----------------------------------------------------------
    print("\n[2/7] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/7] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/7] Training GAN on minority class...")
    minority_emb   = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    if n_to_generate <= 0:
        print("      Training pool already balanced or majority-minority; skipping GAN generation.")
        train_emb_gan = train_emb
        train_lbl_gan = train_lbl
    else:
        generator = train_gan(
            minority_emb,
            noise_dim=GAN_NOISE_DIM,
            epochs=GAN_EPOCHS,
            batch_size=GAN_BATCH_SIZE,
            lr=GAN_LR
        )

        synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
        synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

        train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
        train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/7] Class weights (computed on GAN-balanced pool)
    # ----------------------------------------------------------
    print("\n[4/7] Computing class weights...")
    classes           = np.unique(train_lbl_gan)
    class_weights_arr = compute_class_weight(
        class_weight='balanced', classes=classes, y=train_lbl_gan
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
    print(f"      Class weights: {class_weight_dict}")

    # ----------------------------------------------------------
    # [5/7] Ensemble (DANN-NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[5/7] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training DANN neural network...")
        (feature_extractor, classifier_head), _ = semi_supervised_transfer_learning_dann(
            train_emb_gan, train_lbl_gan,
            test_emb,
            class_weight_dict,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            x_tr_t         = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            train_features = feature_extractor(x_tr_t)
            y_train_pred   = classifier_head(train_features).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(
            train_emb_gan, train_lbl_gan, class_weight_dict, sample_weights
        )
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [6/7] Metrics
    # ----------------------------------------------------------
    print("\n[6/7] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (linux-2.6 Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [7/7] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"            : "dann_gan",
        "train"                 : "all_except_linux-2.6",
        "test"                  : TEST_PROJ,
        "n_train_projects"      : int(len(set(proj_all[train_mask]))),
        "n_train_samples"       : int(len(train_lbl)),
        "n_train_gan"           : int(len(train_lbl_gan)),
        "n_test_samples"        : int(len(test_lbl)),
        "n_vuln_train"          : int(train_lbl.sum()),
        "n_vuln_train_gan"      : int(train_lbl_gan.sum()),
        "n_vuln_test"           : int(test_lbl.sum()),
        "n_synthetic_generated" : int(n_to_generate),
        "accuracy"              : round(float(accuracy),  4),
        "precision"             : round(float(precision), 4),
        "recall"                : round(float(recall),    4),
        "f1"                    : round(float(f1),        4),
        "auc"                   : round(float(auc),        4),
        "g_mean"                : round(float(g_mean),    4),
        "pf"                    : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "dann_gan_linux-2.6.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_dann_gan_linux-2.6.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT vim -> Test on vim
DANN Domain Adaptation + GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "vim"
COMBINED_FILE = "../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/dann_gan/vim"

DANN_EPOCHS      = 50
DANN_BATCH_SIZE   = 64
DOMAIN_LOSS_WEIGHT_MAX = 1.0   # max alpha for domain loss ramp

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator (for minority oversampling)
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Gradient Reversal Layer
# =========================================================
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self):
        super(GradientReversalLayer, self).__init__()

    def forward(self, x, alpha):
        return GradientReversalFunction.apply(x, alpha)


# =========================================================
# Feature Extractor (shared trunk)
# =========================================================
class FeatureExtractor(nn.Module):
    def __init__(self, input_dim):
        super(FeatureExtractor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Classifier Head (vulnerability label predictor)
# =========================================================
class ClassifierHead(nn.Module):
    def __init__(self, feature_dim=16):
        super(ClassifierHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# Domain Discriminator
# =========================================================
class DomainDiscriminator(nn.Module):
    def __init__(self, feature_dim=16):
        super(DomainDiscriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# DANN Training (Phase 1: joint classification + domain adaptation)
# =========================================================
def train_dann(feature_extractor, classifier_head, domain_discriminator,
                x_source, y_source, x_target,
                x_val, y_val, class_weights,
                epochs, batch_size, desc):

    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    grl = GradientReversalLayer()

    params = (list(feature_extractor.parameters()) +
              list(classifier_head.parameters()) +
              list(domain_discriminator.parameters()))
    optimizer = torch.optim.Adam(params, lr=1e-5)

    cls_criterion    = nn.BCELoss(reduction='none')
    domain_criterion = nn.BCELoss()

    x_source_t = torch.tensor(x_source, dtype=torch.float32)
    y_source_t = torch.tensor(y_source, dtype=torch.float32)
    x_target_t = torch.tensor(x_target, dtype=torch.float32)

    source_dataset = TensorDataset(x_source_t, y_source_t)
    source_loader  = DataLoader(source_dataset, batch_size=batch_size, shuffle=True)

    n_batches = len(source_loader)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        domain_discriminator.train()

        target_loader = DataLoader(
            TensorDataset(x_target_t), batch_size=batch_size, shuffle=True
        )
        target_iter = iter(target_loader)

        for batch_idx, (xs_batch, ys_batch) in enumerate(source_loader):
            p = (epoch * n_batches + batch_idx) / (epochs * n_batches)
            alpha = DOMAIN_LOSS_WEIGHT_MAX * (2.0 / (1.0 + np.exp(-10 * p)) - 1.0)

            try:
                (xt_batch,) = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                (xt_batch,) = next(target_iter)

            xs_batch = xs_batch.to(DEVICE)
            ys_batch = ys_batch.to(DEVICE)
            xt_batch = xt_batch.to(DEVICE)

            optimizer.zero_grad()

            # --- Classification loss on source ---
            source_features = feature_extractor(xs_batch)
            y_pred = classifier_head(source_features).squeeze()
            weights = torch.where(ys_batch == 1, pos_weight, neg_weight)
            cls_loss = (cls_criterion(y_pred, ys_batch) * weights).mean()

            # --- Domain loss on source + target ---
            target_features = feature_extractor(xt_batch)

            bsz_s = source_features.size(0)
            bsz_t = target_features.size(0)

            source_reversed = grl(source_features, alpha)
            target_reversed = grl(target_features, alpha)

            domain_pred_source = domain_discriminator(source_reversed)
            domain_pred_target = domain_discriminator(target_reversed)

            domain_labels_source = torch.zeros(bsz_s, 1).to(DEVICE)
            domain_labels_target = torch.ones(bsz_t, 1).to(DEVICE)

            domain_loss = (
                domain_criterion(domain_pred_source, domain_labels_source) +
                domain_criterion(domain_pred_target, domain_labels_target)
            )

            total_loss = cls_loss + domain_loss
            total_loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            val_weights  = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (cls_criterion(val_preds, y_val) * val_weights).mean().item()

        print(f"      {desc} | Epoch {epoch+1}/{epochs} - "
              f"cls_loss: {cls_loss.item():.4f}  domain_loss: {domain_loss.item():.4f}  "
              f"alpha: {alpha:.3f}  val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Fine-tuning Loop (Phase 2: pseudo-labeled, classification only)
# =========================================================
def fine_tune_on_pseudo_labels(feature_extractor, classifier_head,
                                dataloader, x_val, y_val,
                                optimizer, criterion, epochs, desc, class_weights):
    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            features    = feature_extractor(x_batch)
            predictions = classifier_head(features).squeeze()
            weights     = torch.where(y_batch == 1, pos_weight, neg_weight)
            loss        = (criterion(predictions, y_batch) * weights).mean()
            loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            weights      = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (criterion(val_preds, y_val) * weights).mean().item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Semi-supervised Transfer Learning with DANN
# =========================================================
def semi_supervised_transfer_learning_dann(x_train, y_train, x_test,
                                            class_weights, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    feature_extractor     = FeatureExtractor(x_train.shape[1]).to(DEVICE)
    classifier_head       = ClassifierHead(feature_dim=16).to(DEVICE)
    domain_discriminator  = DomainDiscriminator(feature_dim=16).to(DEVICE)

    # Phase 1: DANN training on GAN-oversampled source + target domain alignment
    feature_extractor, classifier_head = train_dann(
        feature_extractor, classifier_head, domain_discriminator,
        x_source=x_tr, y_source=y_tr, x_target=x_test,
        x_val=x_val_t, y_val=y_val_t,
        class_weights=class_weights,
        epochs=DANN_EPOCHS,
        batch_size=DANN_BATCH_SIZE,
        desc=f"Iter {iteration} - Phase 1 (DANN)"
    )

    # Pseudo-label the test set (vim)
    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features = feature_extractor(x_test_t.to(DEVICE))
        y_pred        = classifier_head(test_features).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source (GAN) + pseudo-labelled vim
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(
        list(feature_extractor.parameters()) + list(classifier_head.parameters()),
        lr=1e-5
    )
    criterion = nn.BCELoss(reduction='none')

    feature_extractor, classifier_head = fine_tune_on_pseudo_labels(
        feature_extractor, classifier_head,
        dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2",
        class_weights=class_weights
    )

    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features_final = feature_extractor(x_test_t.to(DEVICE))
        y_pred_final = classifier_head(test_features_final).squeeze().cpu().numpy()

    return (feature_extractor, classifier_head), y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, class_weights, sample_weights=None):
    scale_pos_weight = class_weights[1] / class_weights[0]
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (vim Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except vim -> Test on vim (DANN + GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/7] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/7] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (vim)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/7] Normalize
    # ----------------------------------------------------------
    print("\n[2/7] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/7] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/7] Training GAN on minority class...")
    minority_emb   = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    if n_to_generate <= 0:
        print("      Training pool already balanced or majority-minority; skipping GAN generation.")
        train_emb_gan = train_emb
        train_lbl_gan = train_lbl
    else:
        generator = train_gan(
            minority_emb,
            noise_dim=GAN_NOISE_DIM,
            epochs=GAN_EPOCHS,
            batch_size=GAN_BATCH_SIZE,
            lr=GAN_LR
        )

        synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
        synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

        train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
        train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/7] Class weights (computed on GAN-balanced pool)
    # ----------------------------------------------------------
    print("\n[4/7] Computing class weights...")
    classes           = np.unique(train_lbl_gan)
    class_weights_arr = compute_class_weight(
        class_weight='balanced', classes=classes, y=train_lbl_gan
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
    print(f"      Class weights: {class_weight_dict}")

    # ----------------------------------------------------------
    # [5/7] Ensemble (DANN-NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[5/7] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training DANN neural network...")
        (feature_extractor, classifier_head), _ = semi_supervised_transfer_learning_dann(
            train_emb_gan, train_lbl_gan,
            test_emb,
            class_weight_dict,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            x_tr_t         = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            train_features = feature_extractor(x_tr_t)
            y_train_pred   = classifier_head(train_features).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(
            train_emb_gan, train_lbl_gan, class_weight_dict, sample_weights
        )
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [6/7] Metrics
    # ----------------------------------------------------------
    print("\n[6/7] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (vim Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [7/7] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"            : "dann_gan",
        "train"                 : "all_except_vim",
        "test"                  : TEST_PROJ,
        "n_train_projects"      : int(len(set(proj_all[train_mask]))),
        "n_train_samples"       : int(len(train_lbl)),
        "n_train_gan"           : int(len(train_lbl_gan)),
        "n_test_samples"        : int(len(test_lbl)),
        "n_vuln_train"          : int(train_lbl.sum()),
        "n_vuln_train_gan"      : int(train_lbl_gan.sum()),
        "n_vuln_test"           : int(test_lbl.sum()),
        "n_synthetic_generated" : int(n_to_generate),
        "accuracy"              : round(float(accuracy),  4),
        "precision"             : round(float(precision), 4),
        "recall"                : round(float(recall),    4),
        "f1"                    : round(float(f1),        4),
        "auc"                   : round(float(auc),        4),
        "g_mean"                : round(float(g_mean),    4),
        "pf"                    : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "dann_gan_vim.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_dann_gan_vim.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT FFmpeg -> Test on FFmpeg
DANN Domain Adaptation + GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "FFmpeg"
COMBINED_FILE = "../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/dann_gan/FFmpeg"

DANN_EPOCHS      = 50
DANN_BATCH_SIZE   = 64
DOMAIN_LOSS_WEIGHT_MAX = 1.0   # max alpha for domain loss ramp

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator (for minority oversampling)
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Gradient Reversal Layer
# =========================================================
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self):
        super(GradientReversalLayer, self).__init__()

    def forward(self, x, alpha):
        return GradientReversalFunction.apply(x, alpha)


# =========================================================
# Feature Extractor (shared trunk)
# =========================================================
class FeatureExtractor(nn.Module):
    def __init__(self, input_dim):
        super(FeatureExtractor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Classifier Head (vulnerability label predictor)
# =========================================================
class ClassifierHead(nn.Module):
    def __init__(self, feature_dim=16):
        super(ClassifierHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# Domain Discriminator
# =========================================================
class DomainDiscriminator(nn.Module):
    def __init__(self, feature_dim=16):
        super(DomainDiscriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)


# =========================================================
# DANN Training (Phase 1: joint classification + domain adaptation)
# =========================================================
def train_dann(feature_extractor, classifier_head, domain_discriminator,
                x_source, y_source, x_target,
                x_val, y_val, class_weights,
                epochs, batch_size, desc):

    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    grl = GradientReversalLayer()

    params = (list(feature_extractor.parameters()) +
              list(classifier_head.parameters()) +
              list(domain_discriminator.parameters()))
    optimizer = torch.optim.Adam(params, lr=1e-5)

    cls_criterion    = nn.BCELoss(reduction='none')
    domain_criterion = nn.BCELoss()

    x_source_t = torch.tensor(x_source, dtype=torch.float32)
    y_source_t = torch.tensor(y_source, dtype=torch.float32)
    x_target_t = torch.tensor(x_target, dtype=torch.float32)

    source_dataset = TensorDataset(x_source_t, y_source_t)
    source_loader  = DataLoader(source_dataset, batch_size=batch_size, shuffle=True)

    n_batches = len(source_loader)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        domain_discriminator.train()

        target_loader = DataLoader(
            TensorDataset(x_target_t), batch_size=batch_size, shuffle=True
        )
        target_iter = iter(target_loader)

        for batch_idx, (xs_batch, ys_batch) in enumerate(source_loader):
            p = (epoch * n_batches + batch_idx) / (epochs * n_batches)
            alpha = DOMAIN_LOSS_WEIGHT_MAX * (2.0 / (1.0 + np.exp(-10 * p)) - 1.0)

            try:
                (xt_batch,) = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                (xt_batch,) = next(target_iter)

            xs_batch = xs_batch.to(DEVICE)
            ys_batch = ys_batch.to(DEVICE)
            xt_batch = xt_batch.to(DEVICE)

            optimizer.zero_grad()

            # --- Classification loss on source ---
            source_features = feature_extractor(xs_batch)
            y_pred = classifier_head(source_features).squeeze()
            weights = torch.where(ys_batch == 1, pos_weight, neg_weight)
            cls_loss = (cls_criterion(y_pred, ys_batch) * weights).mean()

            # --- Domain loss on source + target ---
            target_features = feature_extractor(xt_batch)

            bsz_s = source_features.size(0)
            bsz_t = target_features.size(0)

            source_reversed = grl(source_features, alpha)
            target_reversed = grl(target_features, alpha)

            domain_pred_source = domain_discriminator(source_reversed)
            domain_pred_target = domain_discriminator(target_reversed)

            domain_labels_source = torch.zeros(bsz_s, 1).to(DEVICE)
            domain_labels_target = torch.ones(bsz_t, 1).to(DEVICE)

            domain_loss = (
                domain_criterion(domain_pred_source, domain_labels_source) +
                domain_criterion(domain_pred_target, domain_labels_target)
            )

            total_loss = cls_loss + domain_loss
            total_loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            val_weights  = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (cls_criterion(val_preds, y_val) * val_weights).mean().item()

        print(f"      {desc} | Epoch {epoch+1}/{epochs} - "
              f"cls_loss: {cls_loss.item():.4f}  domain_loss: {domain_loss.item():.4f}  "
              f"alpha: {alpha:.3f}  val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Fine-tuning Loop (Phase 2: pseudo-labeled, classification only)
# =========================================================
def fine_tune_on_pseudo_labels(feature_extractor, classifier_head,
                                dataloader, x_val, y_val,
                                optimizer, criterion, epochs, desc, class_weights):
    pos_weight = torch.tensor(class_weights[1], dtype=torch.float32).to(DEVICE)
    neg_weight = torch.tensor(class_weights[0], dtype=torch.float32).to(DEVICE)

    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        feature_extractor.train()
        classifier_head.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            features    = feature_extractor(x_batch)
            predictions = classifier_head(features).squeeze()
            weights     = torch.where(y_batch == 1, pos_weight, neg_weight)
            loss        = (criterion(predictions, y_batch) * weights).mean()
            loss.backward()
            optimizer.step()

        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            val_features = feature_extractor(x_val)
            val_preds    = classifier_head(val_features).squeeze()
            weights      = torch.where(y_val == 1, pos_weight, neg_weight)
            val_loss     = (criterion(val_preds, y_val) * weights).mean().item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return feature_extractor, classifier_head


# =========================================================
# Semi-supervised Transfer Learning with DANN
# =========================================================
def semi_supervised_transfer_learning_dann(x_train, y_train, x_test,
                                            class_weights, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    feature_extractor     = FeatureExtractor(x_train.shape[1]).to(DEVICE)
    classifier_head       = ClassifierHead(feature_dim=16).to(DEVICE)
    domain_discriminator  = DomainDiscriminator(feature_dim=16).to(DEVICE)

    # Phase 1: DANN training on GAN-oversampled source + target domain alignment
    feature_extractor, classifier_head = train_dann(
        feature_extractor, classifier_head, domain_discriminator,
        x_source=x_tr, y_source=y_tr, x_target=x_test,
        x_val=x_val_t, y_val=y_val_t,
        class_weights=class_weights,
        epochs=DANN_EPOCHS,
        batch_size=DANN_BATCH_SIZE,
        desc=f"Iter {iteration} - Phase 1 (DANN)"
    )

    # Pseudo-label the test set (FFmpeg)
    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features = feature_extractor(x_test_t.to(DEVICE))
        y_pred        = classifier_head(test_features).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source (GAN) + pseudo-labelled FFmpeg
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(
        list(feature_extractor.parameters()) + list(classifier_head.parameters()),
        lr=1e-5
    )
    criterion = nn.BCELoss(reduction='none')

    feature_extractor, classifier_head = fine_tune_on_pseudo_labels(
        feature_extractor, classifier_head,
        dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2",
        class_weights=class_weights
    )

    feature_extractor.eval()
    classifier_head.eval()
    with torch.no_grad():
        test_features_final = feature_extractor(x_test_t.to(DEVICE))
        y_pred_final = classifier_head(test_features_final).squeeze().cpu().numpy()

    return (feature_extractor, classifier_head), y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, class_weights, sample_weights=None):
    scale_pos_weight = class_weights[1] / class_weights[0]
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (FFmpeg Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except FFmpeg -> Test on FFmpeg (DANN + GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/7] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/7] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (FFmpeg)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/7] Normalize
    # ----------------------------------------------------------
    print("\n[2/7] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/7] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/7] Training GAN on minority class...")
    minority_emb   = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    if n_to_generate <= 0:
        print("      Training pool already balanced or majority-minority; skipping GAN generation.")
        train_emb_gan = train_emb
        train_lbl_gan = train_lbl
    else:
        generator = train_gan(
            minority_emb,
            noise_dim=GAN_NOISE_DIM,
            epochs=GAN_EPOCHS,
            batch_size=GAN_BATCH_SIZE,
            lr=GAN_LR
        )

        synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
        synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

        train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
        train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/7] Class weights (computed on GAN-balanced pool)
    # ----------------------------------------------------------
    print("\n[4/7] Computing class weights...")
    classes           = np.unique(train_lbl_gan)
    class_weights_arr = compute_class_weight(
        class_weight='balanced', classes=classes, y=train_lbl_gan
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
    print(f"      Class weights: {class_weight_dict}")

    # ----------------------------------------------------------
    # [5/7] Ensemble (DANN-NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[5/7] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training DANN neural network...")
        (feature_extractor, classifier_head), _ = semi_supervised_transfer_learning_dann(
            train_emb_gan, train_lbl_gan,
            test_emb,
            class_weight_dict,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        feature_extractor.eval()
        classifier_head.eval()
        with torch.no_grad():
            x_tr_t         = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            train_features = feature_extractor(x_tr_t)
            y_train_pred   = classifier_head(train_features).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(
            train_emb_gan, train_lbl_gan, class_weight_dict, sample_weights
        )
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [6/7] Metrics
    # ----------------------------------------------------------
    print("\n[6/7] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (FFmpeg Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [7/7] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"            : "dann_gan",
        "train"                 : "all_except_FFmpeg",
        "test"                  : TEST_PROJ,
        "n_train_projects"      : int(len(set(proj_all[train_mask]))),
        "n_train_samples"       : int(len(train_lbl)),
        "n_train_gan"           : int(len(train_lbl_gan)),
        "n_test_samples"        : int(len(test_lbl)),
        "n_vuln_train"          : int(train_lbl.sum()),
        "n_vuln_train_gan"      : int(train_lbl_gan.sum()),
        "n_vuln_test"           : int(test_lbl.sum()),
        "n_synthetic_generated" : int(n_to_generate),
        "accuracy"              : round(float(accuracy),  4),
        "precision"             : round(float(precision), 4),
        "recall"                : round(float(recall),    4),
        "f1"                    : round(float(f1),        4),
        "auc"                   : round(float(auc),        4),
        "g_mean"                : round(float(g_mean),    4),
        "pf"                    : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "dann_gan_FFmpeg.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_dann_gan_FFmpeg.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()